In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
from tqdm import tqdm
from bosperrus import *
plt.rcParams['svg.fonttype'] = 'none'
#plt.rcParams['figure.facecolor'] = "#D9D9D9"

In [ ]:
with open("../../fit_palette.json") as f:
    pal = json.load(f)

In [4]:
np.random.seed(41)

In [5]:
measure = "closeness"

In [6]:
with open("/data/bionets/je30bery/truncated_graphs/mibitof_coords/coords.pickle", "rb") as f:
    datasets = pickle.load(f)

In [7]:
dataset = "TNBC_mibitof:p14_labeledcellData.tiff"
coords = datasets[dataset]

In [8]:
coords = coords[(coords[:, 0] < 1500) & (coords[:,1] < 1500)]
subset = coords /  (np.max(coords, axis=0) - np.min(coords, axis=0)) # [(coords[:, 0] < 1000) & (coords[:,1] < 1000)]

In [9]:
G = nx.Graph()
G.add_nodes_from(range(len(subset)))
for n in G.nodes:
    G.nodes[n]["pos"] = subset[n]
edges = delaunay_edges(subset)
G.add_edges_from(edges)

In [10]:
df =  pd.DataFrame(compute_centrality_measures(edges, len(subset), measures=[measure]))
df["distance"] = distance_to_convex_hull(subset)

In [11]:
d = df["distance"]
C_true = df[measure].values   

In [ ]:
nx.draw(G, pos=nx.get_node_attributes(G, "pos"), node_color=C_true, node_size=3)
plt.axis("square")
plt.savefig("../../result_plots/figure1/fig1_closeness.svg")

In [ ]:
nx.draw(G, pos=nx.get_node_attributes(G, "pos"), node_color=d, node_size=3)
plt.axis("square")
plt.savefig("../../result_plots/figure1/fig1_distance.svg")

In [ ]:
plt.scatter(d[::5], C_true[::5], label="C_true", s=1, color="gray")

for fit in ConstantFit, PiecewiseLinearFit, MichaelisMentenFit, ExponentialSaturationFit:
    f = fit(S_true=C_true, d=d)
    f.fit()
    plt.scatter(d[::5], f.S_model[::5], label=f.name, s=1, color=pal[f.name])
plt.axis("off")
plt.savefig("../../result_plots/figure1/fig1_fits.svg")

In [ ]:
f = Flow.from_distances_and_scores(distances=d, scores=df[["closeness"]])
f.flow()
plt.scatter(d[::5], C_true[::5], label="Closeness", s=1, color="gray", alpha=0.8)
plt.scatter(d[::5], f.observations["BOSPERRUS corrected closeness"][::5], label="BOSPERRUS corrected closeness", s=1, color=pal[f.best_fits["closeness"].name], alpha=0.8)
plt.axis("off")
plt.savefig("../../result_plots/figure1/fig1_corrections.svg")

In [ ]:
nx.draw(G, pos=nx.get_node_attributes(G, "pos"), node_color=f.observations["BOSPERRUS corrected closeness"], node_size=3)
plt.axis("square")
plt.savefig("../../result_plots/figure1/fig1_corrected_closeness.svg")